# sr-diffusion x4 Colab demo

This notebook runs the public `jwheo/sr-diffusion` x4 super-resolution checkpoints from Hugging Face.

Use **Runtime -> Change runtime type -> GPU** before running. The checkpoints are research prototypes under a non-commercial license, not polished production SR models.

Current public inference choices:

- `photo100k_v2_stage4`: recommended default for denoise/sharpening review. It uses the stronger `photo_v2` degradation track and Stage 4 condition-start checkpoint.
- `photo100k_stage4`: milder photo100k Stage 4 checkpoint. It can preserve cleaner inputs better, but is less focused on heavy degradation.
- `prototype_stage4`: older 10k Stage 4 prototype, kept for quick comparison.
- `photo100k_v2_stage3`: previous v2 Stage 3 checkpoint, kept for ablation/comparison.

The newer Stage 2 XL condition encoder has been selected separately, but the XL diffusion checkpoint is not included here until that run is completed and evaluated.


In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU runtime is required. In Colab, use Runtime -> Change runtime type -> GPU.')


## 1. Clone the repo and install lightweight dependencies

Colab already provides PyTorch, so this avoids reinstalling `torch`.


In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir('/content')
repo = Path('sr-diffusion')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/BitIntx/sr-diffusion.git'], check=True)
os.chdir(repo)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('repo:', Path.cwd())


In [ ]:
%pip -q install pyyaml pillow huggingface_hub
%pip -q install -e . --no-deps


## 2. Select a model and download checkpoints

The default downloads only the files needed for the selected variant. Use `photo100k_v2_stage4` first for the current public denoise/sharpening checkpoint.


In [ ]:
import subprocess
from pathlib import Path

REPO_ID = 'jwheo/sr-diffusion'
MODEL_VARIANT = 'photo100k_v2_stage4'  # 'photo100k_v2_stage4', 'photo100k_stage4', 'prototype_stage4', or 'photo100k_v2_stage3'

COMMON_FILES = [
    'LICENSE',
    'CHECKPOINT_LICENSE.md',
    'checkpoints/stage1_autoencoder_best_eval_recon.pt',
]

VARIANTS = {
    'photo100k_v2_stage4': {
        'config': 'configs/hf/diffusion_photo100k_stage4_condition_v2.yaml',
        'files': [
            'checkpoints/stage2_photo100k_v2_b64_best_eval_latent.pt',
            'checkpoints/stage4_photo100k_condition_v2_b32_best_eval_condition_decoded.pt',
        ],
        'note': 'Recommended public denoise/sharpening checkpoint: photo100k v2 Stage 4 condition-start.',
    },
    'photo100k_stage4': {
        'config': 'configs/hf/diffusion_photo100k_stage4_condition.yaml',
        'files': [
            'checkpoints/stage2_photo100k_b64_best_eval_latent.pt',
            'checkpoints/stage4_photo100k_condition_b32_best_eval_condition_decoded.pt',
        ],
        'note': 'Milder photo100k Stage 4 condition-start checkpoint.',
    },
    'prototype_stage4': {
        'config': 'configs/hf/diffusion_stage4_condition.yaml',
        'files': [
            'checkpoints/stage2_latent_pretrain_best_eval_latent.pt',
            'checkpoints/stage4_condition_b32_best_eval_condition_decoded.pt',
        ],
        'note': 'Older 10k Stage 4 condition-start prototype.',
    },
    'photo100k_v2_stage3': {
        'config': 'configs/hf/diffusion_photo100k_v2.yaml',
        'files': [
            'checkpoints/stage2_photo100k_v2_b64_best_eval_latent.pt',
            'checkpoints/stage3_photo100k_v2_b32_best_eval_noise.pt',
        ],
        'note': 'Previous v2 Stage 3 checkpoint, kept for comparison.',
    },
}

selected = VARIANTS[MODEL_VARIANT]
files_to_download = [*COMMON_FILES, *selected['files']]
cmd = ['python', 'scripts/download_hf_checkpoints.py', '--repo-id', REPO_ID]
for filename in files_to_download:
    cmd += ['--file', filename]

print('variant:', MODEL_VARIANT)
print(selected['note'])
print('config:', selected['config'])
subprocess.run(cmd, check=True)

missing = [filename for filename in files_to_download if not Path(filename).exists()]
if missing:
    raise FileNotFoundError(f'Missing downloaded files: {missing}')
CONFIG_PATH = selected['config']
if not Path(CONFIG_PATH).exists():
    raise FileNotFoundError(CONFIG_PATH)
print('downloaded files:', len(files_to_download))


## 3. Choose an input image

Default mode creates a deterministic 512px HR smoke-test image, degrades it with the selected model config, and runs x4 SR. This makes the notebook runnable without uploading anything.

For your own image, set `USE_UPLOAD = True`.

- `INPUT_MODE = 'hr'`: upload or generate an HR image; the script crops/degrades it first for controlled testing.
- `INPUT_MODE = 'lr'`: upload a real LR image. Use tiling for LR images larger than 128x128.


In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter
import numpy as np

USE_UPLOAD = False
INPUT_MODE = 'hr'  # 'hr' for controlled HR->LR->SR smoke test, or 'lr' for a real LR input

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    INPUT_IMAGE = next(iter(uploaded))
else:
    demo_dir = Path('demo_inputs')
    demo_dir.mkdir(exist_ok=True)
    hr_path = demo_dir / 'synthetic_hr_512.png'
    h = w = 512
    yy, xx = np.mgrid[0:h, 0:w]
    base = np.zeros((h, w, 3), dtype=np.uint8)
    base[..., 0] = np.clip(40 + xx * 180 / w, 0, 255)
    base[..., 1] = np.clip(30 + yy * 190 / h, 0, 255)
    base[..., 2] = np.clip(180 - (xx + yy) * 70 / (w + h), 0, 255)
    image = Image.fromarray(base, mode='RGB')
    draw = ImageDraw.Draw(image)
    for i in range(24, 512, 48):
        draw.line((i, 20, 512 - i // 3, 492), fill=(245, 245, 245), width=2)
        draw.rectangle((i, i // 2, i + 38, i // 2 + 38), outline=(20, 20, 20), width=2)
    draw.text((24, 24), 'sr-diffusion smoke test', fill=(255, 255, 255))
    image = image.filter(ImageFilter.UnsharpMask(radius=1.0, percent=120, threshold=3))
    image.save(hr_path)
    if INPUT_MODE == 'hr':
        INPUT_IMAGE = str(hr_path)
    else:
        lr_path = demo_dir / 'synthetic_lr_128.png'
        image.resize((128, 128), Image.Resampling.BICUBIC).save(lr_path)
        INPUT_IMAGE = str(lr_path)

print('input mode:', INPUT_MODE)
print('input:', INPUT_IMAGE)


## 4. Run x4 SR

For Colab T4/L4, keep `STEPS = 32` and `TILE_BATCH_SIZE = 1`. For quick smoke testing, set `STEPS = 8`; quality will be lower.


In [ ]:
import subprocess
from pathlib import Path

USE_TILING = True  # only used with INPUT_MODE='lr'
TILE_OVERLAP = 32
TILE_BATCH_SIZE = 1
OUTPUT_DIR = Path(f'outputs/colab_demo_{MODEL_VARIANT}')
STEPS = 32
SEED = 123

input_flag = '--input-lr' if INPUT_MODE == 'lr' else '--input-hr'
cmd = [
    'python', 'infer_diffusion.py',
    '--config', CONFIG_PATH,
    input_flag, INPUT_IMAGE,
    '--output-dir', str(OUTPUT_DIR),
    '--steps', str(STEPS),
    '--seed', str(SEED),
]
if INPUT_MODE == 'lr' and USE_TILING:
    cmd += ['--tile', '--tile-overlap', str(TILE_OVERLAP), '--tile-batch-size', str(TILE_BATCH_SIZE)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 5. View and download the result

In [ ]:
from IPython.display import display
from PIL import Image
from pathlib import Path

out = Path(f'outputs/colab_demo_{MODEL_VARIANT}')
print('variant:', MODEL_VARIANT)
print('LR input')
display(Image.open(out / 'input_lr.png'))
gt = out / 'gt_hr.png'
if gt.exists():
    print('GT HR')
    display(Image.open(gt))
print('SR output')
display(Image.open(out / 'sr_00.png'))


In [ ]:
from google.colab import files
files.download(str(Path(f'outputs/colab_demo_{MODEL_VARIANT}') / 'sr_00.png'))


## Optional: compare variants

Change `MODEL_VARIANT` in section 2 and rerun sections 2, 4, and 5.

For a current public denoise/sharpening checkpoint, start with `photo100k_v2_stage4`. For cleaner inputs, compare it against `photo100k_stage4`. The v2 checkpoint can suppress stronger degradation better, but may still introduce color/contrast overshoot or small cyan/green sampling artifacts on some images. The XL diffusion run is intentionally not exposed in this demo until it finishes and is evaluated.
